# RT-DETRv4 End-to-End Training Pipeline

In [ ]:
!git clone https://github.com/lyuwenyu/RT-DETR.git
%cd RT-DETR
!pip install -r requirements.txt

In [ ]:
import os, json, shutil
from sklearn.model_selection import train_test_split

BASE_DATA = "/content/ai09-project01/sprint_ai_project1_data"
IMG_DIR = os.path.join(BASE_DATA, "train_images")
ANN_DIR = os.path.join(BASE_DATA, "train_annotations")

CLS_DIR = "/content/extracted/sorted_cls"

OUTPUT_BASE = "/content/rtdetr_dataset"
TRAIN_DIR = os.path.join(OUTPUT_BASE, "train/images")
VAL_DIR = os.path.join(OUTPUT_BASE, "val/images")

os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(VAL_DIR, exist_ok=True)

In [ ]:
records = []

for file in os.listdir(ANN_DIR):
    if file.endswith('.json'):
        with open(os.path.join(ANN_DIR, file)) as f:
            data = json.load(f)
            records.append(data)

print(f"Loaded {len(records)} annotations")

In [ ]:
train_records, val_records = train_test_split(records, test_size=0.2, random_state=42)

print(len(train_records), len(val_records))

In [ ]:
category_set = set()

for r in records:
    for ann in r['annotations']:
        category_set.add(ann['label'])

categories = sorted(list(category_set))
cat2id = {c:i for i,c in enumerate(categories)}

print(len(categories), "classes")

In [ ]:
def convert_to_coco(records, save_path):
    images = []
    annotations = []
    ann_id = 1

    for img_id, r in enumerate(records):
        images.append({
            'id': img_id,
            'file_name': r['image_name'],
            'width': r['width'],
            'height': r['height']
        })

        for ann in r['annotations']:
            x, y, w, h = ann['bbox']
            annotations.append({
                'id': ann_id,
                'image_id': img_id,
                'category_id': cat2id[ann['label']],
                'bbox': [x, y, w, h],
                'area': w*h,
                'iscrowd': 0
            })
            ann_id += 1

    coco = {
        'images': images,
        'annotations': annotations,
        'categories': [{'id': i, 'name': c} for c,i in cat2id.items()]
    }

    with open(save_path, 'w') as f:
        json.dump(coco, f)

In [ ]:
def copy_images(records, dest):
    for r in records:
        src = os.path.join(IMG_DIR, r['image_name'])
        dst = os.path.join(dest, r['image_name'])
        if os.path.exists(src):
            shutil.copy(src, dst)

copy_images(train_records, TRAIN_DIR)
copy_images(val_records, VAL_DIR)

In [ ]:
train_json = os.path.join(OUTPUT_BASE, 'train.json')
val_json = os.path.join(OUTPUT_BASE, 'val.json')

convert_to_coco(train_records, train_json)
convert_to_coco(val_records, val_json)

print("COCO dataset ready")

In [ ]:
config_text = f"""
num_classes: {len(categories)}

train_dataloader:
  dataset:
    img_folder: {TRAIN_DIR}
    ann_file: {train_json}

val_dataloader:
  dataset:
    img_folder: {VAL_DIR}
    ann_file: {val_json}
"""

with open("configs/rtdetr/custom.yaml", "w") as f:
    f.write(config_text)

print("Config created")

In [ ]:
!python tools/train.py -c configs/rtdetr/custom.yaml --use_amp --eval